Привет, ревьювер!

In [1]:
import torch
import pandas as pd
from transformers import AutoTokenizer
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA GeForce GTX 1650


In [2]:
from src.data_utils import preprocess_dataset, split_dataset
import os
import urllib.request

url = "https://code.s3.yandex.net/deep-learning/tweets.txt"
# создаем папку data если нет


raw_path = "data/raw_dataset.csv"
processed_path = "data/dataset_processed.csv"
data_dir = "data"

os.makedirs(data_dir, exist_ok=True)

if not os.path.exists(raw_path):    
    with urllib.request.urlopen(url) as response:
        lines = response.read().decode("utf-8").splitlines()

    df = pd.DataFrame(lines, columns=["text"])
    df.to_csv(raw_path, index=False)

# Очистка
if not os.path.exists(processed_path):
    preprocess_dataset(raw_path, processed_path)

# Разбиение
if not os.path.exists("data/train.csv"):
    split_dataset(processed_path, data_dir)

In [3]:
tokenizer_lstm = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
from torch.utils.data import DataLoader
from src.next_token_dataset import NextTokenDataset, collate_fn

train_df = pd.read_csv("data/train.csv")
val_df = pd.read_csv("data/val.csv")
test_df = pd.read_csv("data/test.csv")

train_df = train_df.dropna()
val_df = val_df.dropna()
test_df = test_df.dropna()

train_ids = tokenizer_lstm(train_df["text"].tolist(), add_special_tokens=False)["input_ids"]
val_ids = tokenizer_lstm(val_df["text"].tolist(), add_special_tokens=False)["input_ids"]
test_ids = tokenizer_lstm(test_df["text"].tolist(), add_special_tokens=False)["input_ids"]

train_dataset = NextTokenDataset(train_ids)
val_dataset = NextTokenDataset(val_ids)
test_dataset = NextTokenDataset(test_ids)

batch_size = 128

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=lambda x: collate_fn(x, tokenizer_lstm.pad_token_id)
)

In [5]:
import torch.nn as nn
from src.lstm_model import LSTMModel

lstm_model = LSTMModel(
    vocab_size=tokenizer_lstm.vocab_size,
    embed_dim=128,
    hidden_dim=256,
    num_layers=1,
    dropout=0.3
).to(device)

criterion = nn.CrossEntropyLoss(ignore_index=tokenizer_lstm.pad_token_id)
optimizer = torch.optim.Adam(lstm_model.parameters(), lr=0.001)

In [6]:
from src.lstm_train import train
n_epoch = 5
train(lstm_model, n_epoch, criterion, optimizer, tokenizer_lstm, train_loader, val_loader, device)
torch.save(lstm_model.state_dict(), "models/lstm_model.pth")

Epoch 1 | Train Loss: 10.2696 | Val Loss: 10.1044 | ROUGE-1: 0.0000 | ROUGE-2: 0.0000

=== Examples ===

Prompt:
nice hair cut dude why were your students leaving in the
Target:
middle of class 1st
Generated:
##izationiciencies arms [unused532]
--------------------------------------------------

Prompt:
couple of beers then home in time to watch labour crumble some moreit may be nearly
Target:
monday but some things warm the cock
Generated:
pork [unused372] tug winning delegatesading ufo
--------------------------------------------------
Epoch 2 | Train Loss: 9.3899 | Val Loss: 8.2641 | ROUGE-1: 0.0058 | ROUGE-2: 0.0000

=== Examples ===

Prompt:
nice hair cut dude why were your students leaving in the
Target:
middle of class 1st
Generated:
everyone isn officials sardinia
--------------------------------------------------

Prompt:
couple of beers then home in time to watch labour crumble some moreit may be nearly
Target:
monday but some things warm the cock
Generated:
##coreganomusm ac

In [7]:
import pandas as pd
from src.eval_transformer_pipeline import (
    build_transformer,
    evaluate_transformer,
    show_example_models
	)
# создаем transformer
generator, tokenizer = build_transformer()

# считаем ROUGE
rouge1, rouge2 = evaluate_transformer(generator, tokenizer, val_df)

print(f"Transformer ROUGE-1: {rouge1:.4f} | ROUGE-2: {rouge2:.4f}")

# показываем примеры
show_example_models(
    generator,
    tokenizer,
    test_df,
    lstm_model,
    tokenizer_lstm,
    num_examples=5
)

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Transformer ROUGE-1: 0.0735 | ROUGE-2: 0.0121

===== Примеры на тестовой выборке =====

Промпт:
glad to hear that u landed

Цель:
safely in us

LSTM:
tot still breathe

TRANSFORMER:
in the middle of

------------------------------------------------------------

Промпт:
thanks to youthat animatedquotmy computer lovequot is in my head good thing i like the

Цель:
song how are ya 2day michelle

LSTM:
he a b and amp your perfect see

TRANSFORMER:
idea of everything at once:

------------------------------------------------------------

Промпт:
alright going down to seattle to check out the new studio and see the

Цель:
guys from this providence peace

LSTM:
and oh you was i know in

TRANSFORMER:
rest of the arena.

------------------------------------------------------------

Промпт:
lol i didn't think you'd be happy i am ok i admit h8ed it 1st but 4th viewing

Цель:
now amp i freaking love it

LSTM:
##nie you ones college in all

TRANSFORMER:
didnt really notice how different this

------

Выводы:
Задача стояла создать модель, которая будет предсказывать тексты и которую можно запускать на мобильных устройствах.  
Сделал модель, попробовала запустить ее на своем слабеньком ноуте на маленьком датасете (10К записей)  - LSTM сильно проигрывает (5 эпох обучения).
Train_loss 6.66 → 3.65 (монотонно падает)
Var_loss 6.02 → 6.23  (сначала чуть падает, потом растёт)
ROUGE-1 0.09–0.11 (медленно становится выше)
ROUGE-2 0.007–0.014 (медленно становится выше, но в целом крайне низкое)
На 10K датасете LSTM выдает случайные фразы, без какого либо контекста.

Понадеялась, что на большом датасете (500К записей) LSTM станет более осмысленной (5 эпох обучения). Но разница только  в значениях loss
Train_loss 5.64 → 4.67
Var_loss 5.33 → 5.08
Rouge1 0.07 → 0,09 → 0,07
Rouge2 0.001 → 0,014
Также генерирует продолжение иногда попадающее по контексту, но складывается ощущение, что случайно

Transformer генерирует более естественные продолжения, хотя Rouge не сильно отличаются от LSTM
Rouge1 0.07
Rouge2 0

Пример генерации lstm на малом датасете
Промт
glad to hear that u landed
Цель:
safely in us
LSTM:
tot still breathe

Пример генерации lstm на большом датасете
Промпт:
please check this pic and comment me in a
Цель:
magazine of my city
LSTM:
time to get a new

Пример генерации DistilGPT2
Промпт:
please check this pic and comment me in a
Цель:
magazine of my city
TRANSFORMER:
comment or like on Facebook
Здесь мы видим более вариативные продолжения, лучше понимает контекст, меньше шаблонных повторов

Итог - моя модель не «понимает» смысл — она продолжает наиболее вероятные куски текста, которые часто встречались в обучении. Хотя в теории сказано что lstm модели можно научить понимать небольшой контекст, и она подходит для задачи автодополнения текста, особенно для мобильных телефонов. 

В общем отрицательный результат - тоже результат. Пока что буду советовать Transformer и разбираться дальше с этой темой.